# Expedia Hotel Recommendations — EDA for DWH staging and product metrics

This notebook is designed to run against the full Expedia `train` dataset without loading all ~37.7M rows into pandas.

**Main approach:** DuckDB scans Parquet/CSV; only aggregated results are fetched into pandas for display/plots.

Goals:

1. reproduce the important data-quality checks;
2. distinguish rows vs events weighted by `cnt`;
3. add metric-oriented EDA;
4. test session definitions;
5. produce a staging-quality specification;
6. collect destination/user hypotheses worth carrying into marts.

The notebook never modifies the source file.

## 0. Data contract that matters

`cnt` is documented as the **number of similar events in the context of the same user session**.

Therefore:

- `COUNT(*)` = aggregated rows;
- `SUM(cnt)` = approximate underlying event count;
- neither is the number of sessions;
- sessionization below is a reconstruction from user timelines.

In [ ]:
# Run this cell once if DuckDB is not installed in the current environment.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])

print("Dependencies are ready.")

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("train.parquet")     # change to train.csv if needed
DESTINATIONS_PATH = Path("destinations.parquet")  # optional
SESSION_USER_SAMPLE_PCT = 5          # 5% of users for expensive session analysis
SESSION_THRESHOLDS_MIN = [15, 30, 60, 120]

assert DATA_PATH.exists(), f"File not found: {DATA_PATH.resolve()}"

con = duckdb.connect()

In [ ]:
def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

def source_expr(path: Path) -> str:
    p = sql_path(path)
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return f"read_parquet('{p}')"
    if suffix in {".csv", ".gz"} or path.name.lower().endswith(".csv.gz"):
        return f"read_csv_auto('{p}', header=true, sample_size=-1)"
    raise ValueError("Use .parquet, .csv or .csv.gz")

SRC = source_expr(DATA_PATH)

def q(query: str) -> pd.DataFrame:
    return con.execute(query).fetchdf()

print("Source:", DATA_PATH.resolve())

## 1. Schema, volume and grain

The first table should answer four different questions:

- how many stored rows;
- how many users;
- how many events after `cnt`;
- how much activity is booking vs click.

In [ ]:
q(f'''
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT user_id) AS users,
    SUM(cnt) AS weighted_events,
    SUM(CASE WHEN is_booking = 1 THEN 1 ELSE 0 END) AS booking_rows,
    SUM(CASE WHEN is_booking = 0 THEN 1 ELSE 0 END) AS click_rows,
    SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) AS booking_events,
    SUM(CASE WHEN is_booking = 0 THEN cnt ELSE 0 END) AS click_events,
    ROUND(100.0 * SUM(CASE WHEN is_booking = 1 THEN 1 ELSE 0 END) / COUNT(*), 3) AS booking_row_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) / SUM(cnt), 3) AS booking_event_rate_pct
FROM {SRC}
''')

In [ ]:
cnt_summary = q(f'''
SELECT
    is_booking,
    COUNT(*) AS rows,
    SUM(cnt) AS weighted_events,
    AVG(cnt) AS mean_cnt,
    approx_quantile(cnt, 0.50) AS p50_cnt,
    approx_quantile(cnt, 0.95) AS p95_cnt,
    approx_quantile(cnt, 0.99) AS p99_cnt,
    MAX(cnt) AS max_cnt
FROM {SRC}
GROUP BY is_booking
ORDER BY is_booking
''')
cnt_summary

In [ ]:
cnt_dist = q(f'''
SELECT
    cnt,
    COUNT(*) AS rows
FROM {SRC}
GROUP BY cnt
ORDER BY cnt
LIMIT 30
''')

ax = cnt_dist.plot(x="cnt", y="rows", kind="bar", legend=False, figsize=(12, 4))
ax.set_title("Distribution of cnt, first 30 values")
ax.set_xlabel("cnt")
ax.set_ylabel("rows")
plt.tight_layout()
plt.show()

### Interpretation

If row-level and event-weighted booking rates differ materially, every downstream metric must expose its denominator explicitly.

## 2. Missing values and exact duplicates

In [ ]:
schema = q(f"DESCRIBE SELECT * FROM {SRC}")
columns = schema["column_name"].tolist()

missing_expr = ",\n".join(
    [f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in columns]
)

missing_counts = q(f'''
SELECT COUNT(*) AS total_rows, {missing_expr}
FROM {SRC}
''')

total_rows = int(missing_counts.loc[0, "total_rows"])

missing = (
    missing_counts.drop(columns="total_rows")
    .T.rename(columns={0: "missing_count"})
    .reset_index(names="column")
)
missing["missing_pct"] = 100 * missing["missing_count"] / total_rows
missing.sort_values("missing_pct", ascending=False)

In [ ]:
q(f"""
SELECT
    total_rows,
    distinct_rows,
    total_rows - distinct_rows AS exact_duplicate_rows
FROM (
    SELECT
        (SELECT COUNT(*) FROM {SRC}) AS total_rows,
        (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {SRC})) AS distinct_rows
)
""")

The exact-duplicate query can be one of the slower full-data checks because it must compare all columns. Run it once during profiling; in the production DWH, store the result as a quality test rather than recalculating it in every dashboard.

## 3. Date-quality checks

In [ ]:
date_quality = q(f'''
WITH x AS (
    SELECT
        TRY_CAST(date_time AS TIMESTAMP) AS event_ts,
        TRY_CAST(srch_ci AS DATE) AS checkin_date,
        TRY_CAST(srch_co AS DATE) AS checkout_date
    FROM {SRC}
)
SELECT
    COUNT(*) AS rows,
    SUM(checkin_date IS NULL)::BIGINT AS missing_checkin,
    SUM(checkout_date IS NULL)::BIGINT AS missing_checkout,
    SUM(checkin_date < CAST(event_ts AS DATE))::BIGINT AS checkin_before_event,
    SUM(checkout_date < checkin_date)::BIGINT AS checkout_before_checkin,
    SUM(checkout_date = checkin_date)::BIGINT AS same_day_stay,
    SUM(EXTRACT(YEAR FROM checkin_date) >= 2050)::BIGINT AS extreme_future_checkin,
    SUM(EXTRACT(YEAR FROM checkout_date) >= 2050)::BIGINT AS extreme_future_checkout
FROM x
''')
date_quality

In [ ]:
q(f'''
WITH x AS (
    SELECT
        TRY_CAST(date_time AS TIMESTAMP) AS event_ts,
        TRY_CAST(srch_ci AS DATE) AS checkin_date,
        TRY_CAST(srch_co AS DATE) AS checkout_date
    FROM {SRC}
)
SELECT
    MIN(event_ts) AS min_event_ts,
    MAX(event_ts) AS max_event_ts,
    MIN(checkin_date) AS min_checkin,
    MAX(checkin_date) AS max_checkin,
    MIN(checkout_date) AS min_checkout,
    MAX(checkout_date) AS max_checkout
FROM x
''')

In [ ]:
year_dist = q(f'''
SELECT
    EXTRACT(YEAR FROM TRY_CAST(srch_ci AS DATE))::INTEGER AS checkin_year,
    COUNT(*) AS rows
FROM {SRC}
WHERE TRY_CAST(srch_ci AS DATE) IS NOT NULL
GROUP BY 1
ORDER BY 1
''')
year_dist

### Metric-valid date fields

Do not delete a row just because one date field is invalid. A row with bad `srch_ci` may still be valid for channel/mobile/user activity analysis.

Create metric-specific validity flags later:

- `valid_for_lead_time`
- `valid_for_stay_length`
- `valid_for_party_metrics`

## 4. Party-size quality

In [ ]:
q(f'''
SELECT
    COUNT(*) AS rows,
    SUM(srch_adults_cnt = 0)::BIGINT AS zero_adults,
    SUM(srch_rm_cnt = 0)::BIGINT AS zero_rooms,
    SUM(srch_adults_cnt + srch_children_cnt = 0)::BIGINT AS zero_travelers,
    SUM((srch_adults_cnt = 0) AND is_booking = 1)::BIGINT AS zero_adults_booking_rows,
    SUM((srch_rm_cnt = 0) AND is_booking = 1)::BIGINT AS zero_rooms_booking_rows,
    SUM((srch_adults_cnt + srch_children_cnt = 0) AND is_booking = 1)::BIGINT AS zero_travelers_booking_rows
FROM {SRC}
''')

Suspicious party rows should be flagged, not automatically deleted: some of them are booking rows.

## 5. Cardinality and concentration

In [ ]:
cat_cols = [
    "site_name", "posa_continent", "user_location_country",
    "user_location_region", "user_location_city", "channel",
    "srch_destination_id", "srch_destination_type_id",
    "hotel_continent", "hotel_country", "hotel_market", "hotel_cluster"
]

expr = ",\n".join([f'COUNT(DISTINCT "{c}") AS "{c}"' for c in cat_cols])
cardinality = q(f"SELECT {expr} FROM {SRC}").T.reset_index()
cardinality.columns = ["column", "unique_values"]
cardinality.sort_values("unique_values")

In [ ]:
def concentration_table(column: str, top_n: int = 15):
    return q(f'''
    WITH counts AS (
        SELECT {column} AS value, SUM(cnt) AS events
        FROM {SRC}
        GROUP BY 1
    ),
    total AS (
        SELECT SUM(events) AS total_events FROM counts
    )
    SELECT
        value,
        events,
        ROUND(100.0 * events / total_events, 3) AS event_share_pct
    FROM counts, total
    ORDER BY events DESC
    LIMIT {int(top_n)}
    ''')

concentration_table("srch_destination_id", 15)

## 6. Booking rates by product cuts

For each segment we show both row-based and `cnt`-weighted rates.

In [ ]:
def segment_rates(column: str, min_rows: int = 1000):
    return q(f'''
    SELECT
        {column} AS segment,
        COUNT(*) AS rows,
        SUM(cnt) AS events,
        ROUND(100.0 * AVG(is_booking), 3) AS booking_row_rate_pct,
        ROUND(
            100.0 * SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) / SUM(cnt),
            3
        ) AS booking_event_rate_pct
    FROM {SRC}
    GROUP BY 1
    HAVING COUNT(*) >= {int(min_rows)}
    ORDER BY events DESC
    ''')

segment_rates("is_mobile")

In [ ]:
segment_rates("is_package")

In [ ]:
segment_rates("channel")

In [ ]:
segment_rates("posa_continent")

In [ ]:
segment_rates("hotel_continent")

## 7. Temporal behavior: seasonality, lead time and stay length

In [ ]:
monthly = q(f'''
WITH x AS (
    SELECT
        DATE_TRUNC('month', TRY_CAST(date_time AS TIMESTAMP)) AS month,
        cnt,
        is_booking
    FROM {SRC}
)
SELECT
    month,
    COUNT(*) AS rows,
    SUM(cnt) AS events,
    SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) AS booking_events,
    100.0 * SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) / SUM(cnt) AS booking_event_rate_pct
FROM x
GROUP BY 1
ORDER BY 1
''')

ax = monthly.plot(x="month", y="events", figsize=(12, 4), legend=False)
ax.set_title("Weighted events by month")
ax.set_ylabel("events")
plt.tight_layout()
plt.show()

monthly

In [ ]:
lead_stay = q(f'''
WITH x AS (
    SELECT
        CAST(TRY_CAST(date_time AS TIMESTAMP) AS DATE) AS event_date,
        TRY_CAST(srch_ci AS DATE) AS checkin_date,
        TRY_CAST(srch_co AS DATE) AS checkout_date,
        cnt,
        is_booking
    FROM {SRC}
),
valid AS (
    SELECT
        DATE_DIFF('day', event_date, checkin_date) AS lead_days,
        DATE_DIFF('day', checkin_date, checkout_date) AS stay_nights,
        cnt,
        is_booking
    FROM x
    WHERE checkin_date >= event_date
      AND checkout_date >= checkin_date
)
SELECT
    approx_quantile(lead_days, 0.10) AS lead_p10,
    approx_quantile(lead_days, 0.50) AS lead_p50,
    approx_quantile(lead_days, 0.90) AS lead_p90,
    approx_quantile(lead_days, 0.99) AS lead_p99,
    approx_quantile(stay_nights, 0.10) AS stay_p10,
    approx_quantile(stay_nights, 0.50) AS stay_p50,
    approx_quantile(stay_nights, 0.90) AS stay_p90,
    approx_quantile(stay_nights, 0.99) AS stay_p99
FROM valid
''')
lead_stay

In [ ]:
lead_buckets = q(f'''
WITH x AS (
    SELECT
        DATE_DIFF(
            'day',
            CAST(TRY_CAST(date_time AS TIMESTAMP) AS DATE),
            TRY_CAST(srch_ci AS DATE)
        ) AS lead_days,
        cnt,
        is_booking
    FROM {SRC}
),
b AS (
    SELECT *,
        CASE
            WHEN lead_days < 0 THEN 'invalid'
            WHEN lead_days = 0 THEN '0'
            WHEN lead_days <= 3 THEN '1-3'
            WHEN lead_days <= 7 THEN '4-7'
            WHEN lead_days <= 14 THEN '8-14'
            WHEN lead_days <= 30 THEN '15-30'
            WHEN lead_days <= 60 THEN '31-60'
            WHEN lead_days <= 120 THEN '61-120'
            ELSE '121+'
        END AS lead_bucket
    FROM x
    WHERE lead_days IS NOT NULL
)
SELECT
    lead_bucket,
    COUNT(*) AS rows,
    SUM(cnt) AS events,
    ROUND(100.0 * SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) / SUM(cnt), 3) AS booking_event_rate_pct
FROM b
GROUP BY 1
ORDER BY events DESC
''')
lead_buckets

## 8. User frequency — before RFM

The dataset has no monetary value, so this is **RF**, not true RFM.

In [ ]:
user_freq = q(f'''
WITH u AS (
    SELECT
        user_id,
        MIN(TRY_CAST(date_time AS TIMESTAMP)) AS first_event,
        MAX(TRY_CAST(date_time AS TIMESTAMP)) AS last_event,
        COUNT(DISTINCT CAST(TRY_CAST(date_time AS TIMESTAMP) AS DATE)) AS active_days,
        COUNT(*) AS rows,
        SUM(cnt) AS events,
        SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) AS booking_events,
        COUNT(DISTINCT srch_destination_id) AS destinations
    FROM {SRC}
    GROUP BY user_id
)
SELECT
    COUNT(*) AS users,
    approx_quantile(active_days, 0.50) AS active_days_p50,
    approx_quantile(active_days, 0.90) AS active_days_p90,
    approx_quantile(events, 0.50) AS events_p50,
    approx_quantile(events, 0.90) AS events_p90,
    approx_quantile(booking_events, 0.50) AS booking_events_p50,
    approx_quantile(booking_events, 0.90) AS booking_events_p90,
    approx_quantile(destinations, 0.50) AS destinations_p50,
    approx_quantile(destinations, 0.90) AS destinations_p90,
    SUM(booking_events > 0)::BIGINT AS users_with_booking
FROM u
''')
user_freq

In [ ]:
active_users_month = q(f'''
SELECT
    DATE_TRUNC('month', TRY_CAST(date_time AS TIMESTAMP)) AS month,
    COUNT(DISTINCT user_id) AS active_users,
    COUNT(DISTINCT CASE WHEN is_booking = 1 THEN user_id END) AS bookers
FROM {SRC}
GROUP BY 1
ORDER BY 1
''')

ax = active_users_month.plot(x="month", y=["active_users", "bookers"], figsize=(12, 4))
ax.set_title("Monthly active users and bookers")
ax.set_ylabel("users")
plt.tight_layout()
plt.show()

## 9. Session reconstruction sensitivity

This is intentionally run on a reproducible sample of users because sorting all 37M rows by user/time is one of the heaviest analyses.

The goal is not to declare 30 minutes correct. The goal is to see whether the session metrics are stable across 15/30/60/120 minutes.

In [ ]:
session_sample_filter = f"(hash(user_id) % 100) < {int(SESSION_USER_SAMPLE_PCT)}"

def session_summary(gap_minutes: int):
    return q(f'''
    WITH ordered AS (
        SELECT
            user_id,
            TRY_CAST(date_time AS TIMESTAMP) AS event_ts,
            cnt,
            is_booking,
            srch_destination_id,
            LAG(TRY_CAST(date_time AS TIMESTAMP)) OVER (
                PARTITION BY user_id
                ORDER BY TRY_CAST(date_time AS TIMESTAMP)
            ) AS prev_ts
        FROM {SRC}
        WHERE {session_sample_filter}
    ),
    marked AS (
        SELECT *,
            CASE
                WHEN prev_ts IS NULL THEN 1
                WHEN DATE_DIFF('minute', prev_ts, event_ts) > {int(gap_minutes)} THEN 1
                ELSE 0
            END AS new_session
        FROM ordered
    ),
    numbered AS (
        SELECT *,
            SUM(new_session) OVER (
                PARTITION BY user_id
                ORDER BY event_ts
                ROWS UNBOUNDED PRECEDING
            ) AS session_num
        FROM marked
    ),
    sessions AS (
        SELECT
            user_id,
            session_num,
            MIN(event_ts) AS session_start,
            MAX(event_ts) AS session_end,
            COUNT(*) AS rows,
            SUM(cnt) AS events,
            MAX(is_booking) AS has_booking,
            COUNT(DISTINCT srch_destination_id) AS destinations
        FROM numbered
        GROUP BY user_id, session_num
    )
    SELECT
        {int(gap_minutes)} AS gap_minutes,
        COUNT(*) AS sessions,
        approx_quantile(rows, 0.50) AS rows_p50,
        approx_quantile(events, 0.50) AS events_p50,
        approx_quantile(DATE_DIFF('minute', session_start, session_end), 0.50) AS duration_min_p50,
        ROUND(100.0 * AVG(has_booking), 3) AS booking_session_rate_pct,
        ROUND(100.0 * AVG(destinations > 1), 3) AS multi_destination_session_pct
    FROM sessions
    ''')

session_sensitivity = pd.concat(
    [session_summary(x) for x in SESSION_THRESHOLDS_MIN],
    ignore_index=True
)
session_sensitivity

In [ ]:
ax = session_sensitivity.plot(
    x="gap_minutes",
    y="sessions",
    marker="o",
    figsize=(8, 4),
    legend=False
)
ax.set_title("Session count sensitivity to inactivity threshold")
ax.set_xlabel("inactivity gap, minutes")
ax.set_ylabel("reconstructed sessions")
plt.tight_layout()
plt.show()

### How to choose a first session rule

Look for a plateau:

- if 30→60 minutes barely changes key statistics, a 30-minute rule is defensible;
- if session count/funnel changes sharply across all thresholds, timeout-only sessionization is unstable;
- then add intent continuity such as destination + check-in/out + party fields.

Always version the rule in DWH.

## 10. Booking-without-click checks

The team's EDA found **0 users** who had a booking somewhere in their history but no click rows at all.

The stronger check is session-level: does a reconstructed booking session contain a click?

In [ ]:
GAP_MIN = 30

booking_session_check = q(f'''
WITH ordered AS (
    SELECT
        user_id,
        TRY_CAST(date_time AS TIMESTAMP) AS event_ts,
        is_booking,
        cnt,
        LAG(TRY_CAST(date_time AS TIMESTAMP)) OVER (
            PARTITION BY user_id ORDER BY TRY_CAST(date_time AS TIMESTAMP)
        ) AS prev_ts
    FROM {SRC}
    WHERE {session_sample_filter}
),
marked AS (
    SELECT *,
        CASE
            WHEN prev_ts IS NULL THEN 1
            WHEN DATE_DIFF('minute', prev_ts, event_ts) > {GAP_MIN} THEN 1
            ELSE 0
        END AS new_session
    FROM ordered
),
numbered AS (
    SELECT *,
        SUM(new_session) OVER (
            PARTITION BY user_id ORDER BY event_ts ROWS UNBOUNDED PRECEDING
        ) AS session_num
    FROM marked
),
sessions AS (
    SELECT
        user_id,
        session_num,
        MAX(is_booking = 1)::INTEGER AS has_booking,
        MAX(is_booking = 0)::INTEGER AS has_click
    FROM numbered
    GROUP BY user_id, session_num
)
SELECT
    COUNT(*) AS sessions,
    SUM(has_booking)::BIGINT AS booking_sessions,
    SUM(has_booking = 1 AND has_click = 0)::BIGINT AS booking_sessions_without_click,
    ROUND(
        100.0 * SUM(has_booking = 1 AND has_click = 0) / NULLIF(SUM(has_booking), 0),
        3
    ) AS booking_sessions_without_click_pct
FROM sessions
''')

booking_session_check

Do not force a literal click→booking funnel if many booking sessions have no logged click. The source may aggregate or omit intermediate events.

## 11. Geography/entity consistency checks

These are staging checks that were missing from the original team notebook.

In [ ]:
mapping_checks = q(f'''
SELECT
    (SELECT COUNT(*) FROM (
        SELECT site_name
        FROM {SRC}
        GROUP BY site_name
        HAVING COUNT(DISTINCT posa_continent) > 1
    )) AS site_to_multiple_posa_continents,

    (SELECT COUNT(*) FROM (
        SELECT hotel_market
        FROM {SRC}
        GROUP BY hotel_market
        HAVING COUNT(DISTINCT hotel_country) > 1
    )) AS market_to_multiple_countries,

    (SELECT COUNT(*) FROM (
        SELECT hotel_country
        FROM {SRC}
        GROUP BY hotel_country
        HAVING COUNT(DISTINCT hotel_continent) > 1
    )) AS hotel_country_to_multiple_continents,

    (SELECT COUNT(*) FROM (
        SELECT user_location_city
        FROM {SRC}
        GROUP BY user_location_city
        HAVING COUNT(DISTINCT user_location_country) > 1
            OR COUNT(DISTINCT user_location_region) > 1
    )) AS user_city_to_multiple_parent_geographies
''')
mapping_checks

A non-zero result is not automatically an error: IDs may not be globally hierarchical. But it tells us whether a clean dimension table can safely assume one-to-one parent mappings.

## 12. Destination behavior hypotheses

In [ ]:
top_dest = q(f'''
SELECT
    srch_destination_id,
    COUNT(*) AS rows,
    SUM(cnt) AS events,
    SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) AS booking_events,
    ROUND(100.0 * SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) / SUM(cnt), 3) AS booking_event_rate_pct,
    COUNT(DISTINCT user_id) AS users,
    COUNT(DISTINCT hotel_cluster) AS hotel_clusters
FROM {SRC}
GROUP BY 1
HAVING SUM(cnt) >= 1000
ORDER BY events DESC
LIMIT 30
''')
top_dest

In [ ]:
repeat_destination = q(f'''
WITH ud AS (
    SELECT
        user_id,
        srch_destination_id,
        SUM(cnt) AS events,
        SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) AS booking_events
    FROM {SRC}
    GROUP BY 1, 2
),
u AS (
    SELECT
        user_id,
        COUNT(*) AS destinations,
        SUM(events > 1)::INTEGER AS repeated_destinations,
        SUM(booking_events > 0)::INTEGER AS booked_destinations
    FROM ud
    GROUP BY 1
)
SELECT
    COUNT(*) AS users,
    approx_quantile(destinations, 0.50) AS destinations_p50,
    approx_quantile(destinations, 0.90) AS destinations_p90,
    AVG(repeated_destinations > 0) * 100 AS users_with_repeat_destination_pct,
    AVG(booked_destinations > 1) * 100 AS users_with_multiple_booked_destinations_pct
FROM u
''')
repeat_destination

## 13. Optional `destinations` table coverage

Run only if you have `destinations.parquet` or change `DESTINATIONS_PATH`.

In [ ]:
if DESTINATIONS_PATH.exists():
    DEST_SRC = source_expr(DESTINATIONS_PATH)
    display(q(f'''
    WITH ids AS (
        SELECT DISTINCT srch_destination_id
        FROM {SRC}
    ),
    d AS (
        SELECT DISTINCT srch_destination_id
        FROM {DEST_SRC}
    )
    SELECT
        COUNT(*) AS train_destination_ids,
        SUM(d.srch_destination_id IS NOT NULL)::BIGINT AS covered_ids,
        SUM(d.srch_destination_id IS NULL)::BIGINT AS missing_ids,
        ROUND(100.0 * AVG(d.srch_destination_id IS NOT NULL), 3) AS coverage_pct
    FROM ids
    LEFT JOIN d USING (srch_destination_id)
    '''))
else:
    print("No destinations file found; skipping.")

## 14. Staging quality flags — proposed SQL contract

This query does not write anything. It shows the columns I recommend materializing in STG/CLEAN.

In [ ]:
staging_preview = q(f'''
SELECT
    *,
    TRY_CAST(date_time AS TIMESTAMP) AS event_ts,
    TRY_CAST(srch_ci AS DATE) AS checkin_date,
    TRY_CAST(srch_co AS DATE) AS checkout_date,

    orig_destination_distance IS NULL AS distance_is_missing,
    TRY_CAST(srch_ci AS DATE) IS NULL AS q_missing_checkin,
    TRY_CAST(srch_co AS DATE) IS NULL AS q_missing_checkout,

    TRY_CAST(srch_ci AS DATE) < CAST(TRY_CAST(date_time AS TIMESTAMP) AS DATE)
        AS q_checkin_before_event,

    TRY_CAST(srch_co AS DATE) < TRY_CAST(srch_ci AS DATE)
        AS q_checkout_before_checkin,

    TRY_CAST(srch_co AS DATE) = TRY_CAST(srch_ci AS DATE)
        AS q_same_day_stay,

    srch_adults_cnt = 0 AS q_zero_adults,
    srch_rm_cnt = 0 AS q_zero_rooms,
    srch_adults_cnt + srch_children_cnt = 0 AS q_zero_travelers,

    EXTRACT(YEAR FROM TRY_CAST(srch_ci AS DATE)) >= 2050
        OR EXTRACT(YEAR FROM TRY_CAST(srch_co AS DATE)) >= 2050
        AS q_extreme_future_date
FROM {SRC}
LIMIT 100
''')

staging_preview.head()

## 15. Final checklist

### STG
- preserve raw values;
- cast dates/types;
- add quality flags;
- preserve NULL distance;
- audit duplicate groups.

### CLEAN / SILVER
- dedupe only after audit;
- `lead_days`, `stay_nights`, temporal dimensions;
- metric-valid flags;
- versioned reconstructed `session_id`.

### MARTS
- `mart_sessions`
- `mart_user_activity`
- `mart_user_rf`
- `mart_funnel`
- `mart_destination`
- `mart_origin_destination`
- `mart_data_quality_daily`

### Metrics
Never mix:
- rows;
- `cnt`-weighted events;
- sessions;
- users;
- booking users.

## 16. External-analysis ideas incorporated here

Recurring public-EDA ideas used in this notebook:

- parse and engineer temporal features;
- treat `cnt` as session-context event multiplicity;
- inspect invalid stay dates;
- destination-conditioned behavior;
- destination/hotel geography as high-signal categorical structure;
- preserve semantic missingness for distance;
- avoid relying on simple linear correlation;
- validate with time-aware logic rather than random-only splits.

The exact external links and source caveats are listed in the separate report.